::: {.sb-lesson-links}
[Exercises →](../exercises/ridge-vs-lasso.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/03-regularization.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-workflow.ipynb){download="lesson-workflow.ipynb"}  
Data: [customers.csv](../downloads/data/customers.csv){download="customers.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

## What will you learn in this course? 🧐🧐

Let's come back to the multiple linear regression model of the previous section (predicting the salary of customers). Now, our objective is to implement a **regularized** linear regression. In this process, we'll use :

* A Ridge linear regression model
* cross-validation to estimate how the generalized $R^2$ score varies depending on the choice of the validation set
* cross-validated grid search to tune the value of the regularization strength

It's quite an ambitious program, isn't it ? 🥵🥵

But don't worry, with scikit-learn's dedicated classes, it will be quite easy and straightforward to implement 😌😌

* Training Pipeline
    * Cross-validated score for a Ridge model (with default value of $\lambda$)
    * Grid search : tune $\lambda$
* Test pipeline
* Final remarks

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import r2_score

In [ ]:
# Import dataset
print("Loading dataset...")
dataset = pd.read_csv("../data/customers.csv")
print("...Done.")
dataset.head()

Loading dataset...
...Done.


,Country,Age,Salary,Purchased
0,Germany,20.0,38500.0,No
1,Spain,33.0,32200.0,No
2,Italy,43.0,47900.0,No
3,Germany,62.0,54300.0,No
4,France,53.0,51800.0,Yes


In [ ]:
# Separate target variable Y from features X
print("Separating labels from features...")
target_variable = "Salary"

X = dataset.drop(target_variable, axis = 1)
Y = dataset.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Separating labels from features...
...Done.

Y : 
0    38500.0
1    32200.0
2    47900.0
3    54300.0
4    51800.0
Name: Salary, dtype: float64

X :
   Country   Age Purchased
0  Germany  20.0        No
1    Spain  33.0        No
2    Italy  43.0        No
3  Germany  62.0        No
4   France  53.0       Yes


In [ ]:
# Automatically detect names of numeric/categorical columns
numeric_features = []
categorical_features = []
for i,t in X.dtypes.items():
    if ('float' in str(t)) or ('int' in str(t)) :
        numeric_features.append(i)
    else :
        categorical_features.append(i)

print('Found numeric features ', numeric_features)
print('Found categorical features ', categorical_features)

Found numeric features  ['Age']
Found categorical features  ['Country', 'Purchased']


In [ ]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



### Preprocessing

In [ ]:
# Create pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # missing values will be replaced by columns' mean
    ('scaler', StandardScaler())
])

In [ ]:
# Create pipeline for categorical features
categorical_transformer = Pipeline(
    steps=[
    ('encoder', OneHotEncoder(drop='first')) # first column will be dropped to avoid creating correlations between features
    ])

In [ ]:
# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [ ]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print('...Done.')
print(X_train[0:5]) # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head()) 
X_test = preprocessor.transform(X_test) # Don't fit again !!
print('...Done.')
print(X_test[0:5,:]) # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore
print()

Performing preprocessings on train set...
     Country   Age Purchased
336   France  53.0       Yes
64   Germany  58.0       Yes
55   Germany  59.0        No
106    Spain  38.0        No
300  Germany  42.0        No
...Done.
[[ 0.77179836  0.          0.          0.          1.        ]
 [ 1.14799315  1.          0.          0.          1.        ]
 [ 1.22323211  1.          0.          0.          0.        ]
 [-0.35678602  0.          0.          1.          0.        ]
 [-0.05583018  1.          0.          0.          0.        ]]

Performing preprocessings on test set...
     Country   Age Purchased
132    Spain  31.0        No
309   France  22.0        No
341    Italy  35.0        No
196    Spain  44.0        No
246  Germany  31.0        No
...Done.
[[-0.88345873  0.          0.          1.          0.        ]
 [-1.56060935  0.          0.          0.          0.        ]
 [-0.58250289  0.          1.          0.          0.        ]
 [ 0.09464773  0.          0.          1.    

### Cross-validated score for a Ridge model (with default value of $\lambda$)

In [ ]:
# Perform 3-fold cross-validation to evaluate the generalized R2 score obtained with a Ridge model
print("3-fold cross-validation...")
regressor = Ridge()
scores = cross_val_score(regressor, X_train, Y_train, cv=3)
print('The cross-validated R2-score is : ', scores.mean())
print('The standard deviation is : ', scores.std())

3-fold cross-validation...
The cross-validated R2-score is :  0.7935140964507607
The standard deviation is :  0.02192098700804002


### Grid search : tune $\lambda$

In [ ]:
# Perform grid search
print("Grid search...")
regressor = Ridge()
# Grid of values to be tested
params = {
    'alpha': [0.0, 0.1, 1.0, 10.0, 100.0, 1000.0] # 0 corresponds to no regularization
}
gridsearch = GridSearchCV(regressor, param_grid = params, cv = 3) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best R2 score : ", gridsearch.best_score_)

# mean cross-validated R2 for each value of alpha
pd.DataFrame(gridsearch.cv_results_)[["param_alpha", "mean_test_score", "std_test_score"]]

Grid search...


...Done.
Best hyperparameters :  {'alpha': 1.0}
Best R2 score :  0.7935140964507607


,param_alpha,mean_test_score,std_test_score
0,0.0,0.793075,0.022280
1,0.1,0.793128,0.022239
2,1.0,0.793514,0.021921
3,10.0,0.792367,0.021036
4,100.0,0.708175,0.023281
5,1000.0,0.264339,0.011223


### Performance assessment

In [ ]:
# Print R^2 scores
print("R2 score on training set : ", gridsearch.score(X_train, Y_train))
print("R2 score on test set : ", gridsearch.score(X_test, Y_test))

R2 score on training set :  0.8058436939720461
R2 score on test set :  0.7362819682943258


## Final remarks
Here, regularization barely changes the score: the best values of $\lambda$ give about the same $R^2$ as no regularization at all, and very large values make the model worse (it underfits). This is expected: with **3 features and 320 training examples**, a linear model has no room to overfit, so there is nothing for regularization to fix.

Regularization pays off when the model has **many features compared to the number of examples**. You will see it in the exercise on gene expression data: 2,000 genes for 801 samples, where an unregularized linear regression overfits badly and Lasso finds the few genes that matter 🧬

::: {.sb-next}
**[Your turn: Regularization exercises →](../exercises/ridge-vs-lasso.ipynb)**
:::